# Neural Graph Collaborative Filtering (NGCF) — End-to-End Pipeline (with SGL Preprocess & Evaluation)

This notebook implements **NGCF** from the SIGIR 2019 paper:
> **Neural Graph Collaborative Filtering**  
> Xiang Wang, Xiangnan He, Meng Wang, Fuli Feng, and Tat-Seng Chua.  
> *SIGIR 2019* | [Official Repository (xiangwang1223/neural_graph_collaborative_filtering)](https://github.com/xiangwang1223/neural_graph_collaborative_filtering)

configured according to the **baseline setup reported in the SGL paper** (Section 4.1.2 of Wu et al., SIGIR 2021), while keeping the **exact same data preprocessing, 7:1:2 (`train:val:test`) split from `yelp2018.train`, BPR sampling, and evaluation pipeline** as our SGL and LightGCN notebooks for a fair comparison.

### Experimental Setup (Matching SGL Paper Section 4.1.2 & Official NGCF Code)
- **Embedding Size ($d$)**: `64` (`weight_size = [64, 64, 64]`, 3 propagation layers)
- **Message Construction & Aggregation**:
  $$\mathbf{e}_u^{(k)} = \text{LeakyReLU}\!\left(\mathbf{W}_1^{(k)} \left(\mathbf{e}_u^{(k-1)} + \sum_{i \in \mathcal{N}_u} \tilde{A}_{ui}\,\mathbf{e}_i^{(k-1)}\right) + \mathbf{b}_1^{(k)} + \mathbf{W}_2^{(k)} \left(\mathbf{e}_u^{(k-1)} \odot \sum_{i \in \mathcal{N}_u} \tilde{A}_{ui}\,\mathbf{e}_i^{(k-1)}\right) + \mathbf{b}_2^{(k)}\right)$$
- **Layer Combination**: Concatenation across all $L+1$ layers (`torch.cat(all_embeddings, dim=1)` $\rightarrow$ dimension $64 \times 4 = 256$), L2-normalized per layer (`F.normalize(..., p=2, dim=1)`).
- **Optimization**: Adam (`lr = 0.001`), `batch_size = 2048`, Xavier initialization, `node_dropout = 0.1`, `mess_dropout = [0.1, 0.1, 0.1]`.

### Pipeline Structure
1. **Environment & Reproducibility Setup**: Imports, device auto-detection (CUDA / MPS / CPU), and random seeds.
2. **Data Preprocessing (`yelp2018.train` 7:1:2 Split, Graph Laplacian & BPR Sampler)**:
   - Automatic downloader for **`yelp2018.train` only** from the `SGL-Torch` repository (plus a synthetic fallback).
   - Per-user **7 : 1 : 2 (`train : val : test`) interaction split** (70% training, 10% validation, 20% testing) from `yelp2018.train`.
   - Graph Laplacian built strictly from the 70% training split (supports both `norm_adj` $\mathbf{D}^{-1}(\mathbf{A}+\mathbf{I})$ from official NGCF and symmetric $\mathbf{D}^{-1/2}\mathbf{A}\mathbf{D}^{-1/2}$).
   - Mini-batch BPR pairwise sampler $(u, i, j)$.
3. **Model Architecture (`NGCF` from `NGCF/NGCF.py`)**:
   - Bi-Interaction GCN layers with $\mathbf{W}_1^{(k)}, \mathbf{b}_1^{(k)}$ (GCN part) and $\mathbf{W}_2^{(k)}, \mathbf{b}_2^{(k)}$ (element-wise interaction part), LeakyReLU($0.2$), message dropout, node dropout, and layer concatenation.
4. **Evaluation (`Evaluator`)**:
   - All-item ranking protocol for `Precision@K`, `Recall@K`, and `NDCG@K` on both the **Validation (10%)** and **Test (20%)** splits, plus the 10-group **Long-Tail Item Group Recall** analysis (Section 4.3.1 of the SGL paper).
5. **Training Loop & Visualization**:
   - End-to-end execution using the 10% validation split for early stopping and checkpointing, followed by final evaluation on the 20% test split.

In [ ]:
# ==============================================================================
# 1. Environment Setup & Imports (Colab Ready)
# ==============================================================================
import os
import random
import urllib.request
from collections import defaultdict
from dataclasses import dataclass, field
from time import time

import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F


def set_seed(seed: int = 2021) -> None:
    """Fixes random seeds across Python, NumPy, and PyTorch (CPU & CUDA) for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Running on Device: {DEVICE}")

## 2. Data Preprocessing (`SGLDataset` & `PairwiseSampler`)

This section uses the **exact same data preprocessing pipeline** as our SGL and LightGCN notebooks: loading **`yelp2018.train` only**, splitting each user's interactions into **train : val : test = 7 : 1 : 2** (70% / 10% / 20%), building the sparse graph Laplacian matrix on the 70% training split, and sampling BPR triplets $(u, i, j)$. It supports both the symmetric normalized adjacency $\tilde{\mathbf{A}} = \mathbf{D}^{-1/2}\mathbf{A}\mathbf{D}^{-1/2}$ (`adj_type='sym_adj'`) and the random-walk with self-loop Laplacian $\mathcal{L} = \mathbf{D}^{-1}(\mathbf{A} + \mathbf{I})$ (`adj_type='norm_adj'`, default in `xiangwang1223/neural_graph_collaborative_filtering`).

In [ ]:
# ==============================================================================
# 2. Data Preprocessing: Single-File (yelp2018.train) 7:1:2 Split & BPR Sampler
# ==============================================================================

SGL_DATASET_BASE_URL = "https://raw.githubusercontent.com/wujcan/SGL-Torch/master/dataset"


def download_sgl_dataset(data_dir: str, dataset_name: str = "yelp2018") -> str:
    """Downloads ONLY `<dataset_name>.train` from the official SGL-Torch GitHub repo."""
    os.makedirs(data_dir, exist_ok=True)
    train_path = os.path.join(data_dir, f"{dataset_name}.train")
    if not os.path.exists(train_path):
        url = f"{SGL_DATASET_BASE_URL}/{dataset_name}/{dataset_name}.train"
        print(f"[Dataset] Downloading {dataset_name}.train from {url} ...")
        urllib.request.urlretrieve(url, train_path)
    return train_path


def generate_synthetic_dataset(
    data_dir: str,
    num_users: int = 300,
    num_items: int = 500,
    seed: int = 2021,
) -> str:
    """Generates a power-law synthetic `.train` file (each line: `user_id item_id`)."""
    os.makedirs(data_dir, exist_ok=True)
    rng = np.random.default_rng(seed)

    # Power-law item popularity distribution p(i) ~ (i + 10)^(-0.8)
    ranks = np.arange(1, num_items + 1, dtype=np.float64)
    item_probs = (ranks + 10.0) ** (-0.8)
    item_probs /= item_probs.sum()

    train_path = os.path.join(data_dir, "synthetic.train")
    with open(train_path, "w", encoding="utf-8") as f_train:
        for u in range(num_users):
            n_inter = int(rng.integers(15, 35))
            items = rng.choice(num_items, size=n_inter, replace=False, p=item_probs)
            for i in items:
                f_train.write(f"{u} {int(i)}\n")
    return train_path


def sp_mat_to_sp_tensor(sp_mat: sp.spmatrix, device: torch.device = DEVICE) -> torch.Tensor:
    """
    Converts a SciPy sparse matrix into a coalesced PyTorch sparse_coo_tensor.
    Equivalent to `_convert_sp_mat_to_sp_tensor` in NGCF / SGL-Torch.
    """
    coo = sp_mat.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((coo.row, coo.col)).astype(np.int64))
    values = torch.from_numpy(coo.data)
    shape = torch.Size(coo.shape)
    return torch.sparse_coo_tensor(indices, values, shape, device=device).coalesce()


class SGLDataset:
    """
    Loads `<dataset_name>.train` ONLY (e.g. `yelp2018.train`), splits each user's
    interactions into train : val : test with ratio 7 : 1 : 2 (70% / 10% / 20%),
    and constructs the normalized bipartite adjacency / Laplacian matrix using ONLY
    the 70% training split for NGCF.
    """

    def __init__(
        self,
        data_dir: str = "./dataset/yelp2018",
        dataset_name: str = "yelp2018",
        adj_type: str = "norm_adj",
        split_ratios: tuple[float, float, float] = (0.7, 0.1, 0.2),
        seed: int = 2021,
        device: torch.device = DEVICE,
    ) -> None:
        self.data_dir = data_dir
        self.dataset_name = dataset_name
        self.adj_type = adj_type
        self.split_ratios = split_ratios
        self.seed = seed
        self.device = device

        # 1. Resolve `<dataset_name>.train` path (ignores `.test` completely)
        if dataset_name == "synthetic":
            train_path = os.path.join(data_dir, "synthetic.train")
            if not os.path.exists(train_path):
                train_path = generate_synthetic_dataset(data_dir, seed=seed)
        else:
            train_path = os.path.join(data_dir, f"{dataset_name}.train")
            if not os.path.exists(train_path):
                train_path = download_sgl_dataset(data_dir, dataset_name)

        # 2. Load all raw interactions from `<dataset_name>.train` and split 7:1:2 per user
        (
            self.num_users,
            self.num_items,
            self.total_interactions,
            self.train_users,
            self.train_items,
            self.train_user_dict,
            self.val_user_dict,
            self.test_user_dict,
            self.train_val_user_dict,
            self.train_user_set,
        ) = self._load_and_split_train_file(train_path, split_ratios=split_ratios, seed=seed)

        self.num_nodes = self.num_users + self.num_items
        self.num_ratings = len(self.train_users)
        self.num_val_ratings = sum(len(v) for v in self.val_user_dict.values())
        self.num_test_ratings = sum(len(v) for v in self.test_user_dict.values())

        # 3. Pre-construct the normalized adjacency matrix using ONLY the 70% train split
        self.norm_adj_sp = self.create_adj_mat(
            self.train_users, self.train_items, adj_type=self.adj_type
        )
        self.norm_adj = sp_mat_to_sp_tensor(self.norm_adj_sp, device=self.device)

    @staticmethod
    def _load_and_split_train_file(
        train_path: str,
        split_ratios: tuple[float, float, float] = (0.7, 0.1, 0.2),
        seed: int = 2021,
    ):
        """
        Reads user-item interactions from `<dataset_name>.train` ONLY and splits each
        user's interactions into train : val : test according to `split_ratios` (7:1:2).
        """
        raw_user_items = defaultdict(list)
        max_u, max_i = -1, -1

        with open(train_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(",")
                u = int(parts[0])
                max_u = max(max_u, u)
                for item_str in parts[1:]:
                    i = int(item_str)
                    raw_user_items[u].append(i)
                    max_i = max(max_i, i)

        num_users = max_u + 1
        num_items = max_i + 1

        # Normalize ratios to sum to 1.0 (e.g., (7, 1, 2) or (0.7, 0.1, 0.2) -> 0.7, 0.1, 0.2)
        r_sum = float(sum(split_ratios))
        train_ratio = split_ratios[0] / r_sum
        val_ratio = split_ratios[1] / r_sum

        rng = np.random.default_rng(seed)

        train_users, train_items = [], []
        train_user_dict = {}
        val_user_dict = {}
        test_user_dict = {}
        train_val_user_dict = {}
        train_user_set = defaultdict(set)
        total_interactions = 0

        for u in range(num_users):
            items_list = raw_user_items.get(u, [])
            if not items_list:
                continue
            # Deduplicate while preserving initial order before shuffling
            items = np.array(list(dict.fromkeys(items_list)), dtype=np.int32)
            rng.shuffle(items)
            n = len(items)
            total_interactions += n

            if n >= 3:
                n_train = max(1, int(round(train_ratio * n)))
                n_val = max(1, int(round(val_ratio * n)))
                if n_train + n_val >= n:
                    n_train = max(1, n - 2)
                    n_val = 1
                u_train = items[:n_train]
                u_val = items[n_train : n_train + n_val]
                u_test = items[n_train + n_val :]
            elif n == 2:
                u_train = items[:1]
                u_val = np.array([], dtype=np.int32)
                u_test = items[1:]
            else:
                u_train = items
                u_val = np.array([], dtype=np.int32)
                u_test = np.array([], dtype=np.int32)

            train_user_dict[u] = u_train
            if len(u_val) > 0:
                val_user_dict[u] = u_val
            if len(u_test) > 0:
                test_user_dict[u] = u_test

            # Combined train + val items per user for masking during final Test evaluation
            train_val_user_dict[u] = (
                np.concatenate([u_train, u_val]) if len(u_val) > 0 else u_train
            )

            train_user_set[u] = set(u_train.tolist())
            train_users.extend([u] * len(u_train))
            train_items.extend(u_train.tolist())

        return (
            num_users,
            num_items,
            total_interactions,
            np.asarray(train_users, dtype=np.int32),
            np.asarray(train_items, dtype=np.int32),
            train_user_dict,
            val_user_dict,
            test_user_dict,
            train_val_user_dict,
            train_user_set,
        )

    def create_adj_mat(
        self, users: np.ndarray, items: np.ndarray, adj_type: str = "norm_adj"
    ) -> sp.csr_matrix:
        """
        Constructs the bipartite adjacency matrix following `load_data.py` in the
        official NGCF repository (`xiangwang1223/neural_graph_collaborative_filtering`):
        - 'plain_adj': A
        - 'norm_adj' : D^{-1} * (A + I)  (Default in official NGCF parser.py)
        - 'sym_adj'  : D^{-1/2} * A * D^{-1/2}  (Symmetric Laplacian used in SGL / LightGCN)
        - 'mean_adj' : D^{-1} * A
        """
        user_np = np.array(users)
        item_np = np.array(items) + self.num_users
        ratings = np.ones_like(user_np, dtype=np.float32)

        row = np.concatenate([user_np, item_np])
        col = np.concatenate([item_np, user_np])
        data = np.concatenate([ratings, ratings])
        adj_mat = sp.csr_matrix((data, (row, col)), shape=(self.num_nodes, self.num_nodes))

        def normalized_adj_single(adj: sp. spmatrix) -> sp.csr_matrix:
            rowsum = np.array(adj.sum(1)).squeeze()
            with np.errstate(divide="ignore"):
                d_inv = np.power(rowsum, -1.0).flatten()
            d_inv[np.isinf(d_inv)] = 0.0
            d_mat_inv = sp.diags(d_inv)
            return d_mat_inv.dot(adj).tocsr()

        def sym_normalized_adj(adj: sp.spmatrix) -> sp.csr_matrix:
            rowsum = np.array(adj.sum(1)).squeeze()
            with np.errstate(divide="ignore"):
                d_inv = np.power(rowsum, -0.5).flatten()
            d_inv[np.isinf(d_inv)] = 0.0
            d_mat_inv = sp.diags(d_inv)
            return d_mat_inv.dot(adj).dot(d_mat_inv).tocsr()

        if adj_type == "plain_adj":
            return adj_mat.tocsr()
        elif adj_type == "norm_adj":
            return normalized_adj_single(adj_mat + sp.eye(adj_mat.shape[0], dtype=np.float32))
        elif adj_type == "mean_adj":
            return normalized_adj_single(adj_mat)
        elif adj_type == "sym_adj":
            return sym_normalized_adj(adj_mat)
        else:
            raise ValueError(f"Unknown adj_type: {adj_type}")


class PairwiseSampler:
    """
    Mini-batch sampler for Bayesian Personalized Ranking (BPR) loss.
    For each positive training interaction (u, i) in the 70% train split,
    uniformly samples a negative item j not in u's training history.
    """

    def __init__(
        self, dataset: SGLDataset, batch_size: int = 2048, shuffle: bool = True
    ) -> None:
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_users = dataset.num_users
        self.num_items = dataset.num_items
        self.train_users = dataset.train_users
        self.train_items = dataset.train_items
        self.train_user_set = dataset.train_user_set
        self.num_ratings = len(self.train_users)

    def __len__(self) -> int:
        return math.ceil(self.num_ratings / self.batch_size)

    def __iter__(self):
        indices = np.arange(self.num_ratings)
        if self.shuffle:
            np.random.shuffle(indices)

        users_shuffled = self.train_users[indices]
        pos_items_shuffled = self.train_items[indices]

        # Sample one negative item per positive training interaction
        neg_items = np.empty(self.num_ratings, dtype=np.int32)
        for idx, u in enumerate(users_shuffled):
            pos_set = self.train_user_set[int(u)]
            j = random.randint(0, self.num_items - 1)
            while j in pos_set:
                j = random.randint(0, self.num_items - 1)
            neg_items[idx] = j

        for start in range(0, self.num_ratings, self.batch_size):
            end = min(start + self.batch_size, self.num_ratings)
            yield (
                users_shuffled[start:end],
                pos_items_shuffled[start:end],
                neg_items[start:end],
            )


## 3. NGCF Model Architecture (`xiangwang1223/neural_graph_collaborative_filtering`)

Below is the PyTorch implementation of **NGCF** faithful to [`xiangwang1223/neural_graph_collaborative_filtering/NGCF/NGCF.py`](https://github.com/xiangwang1223/neural_graph_collaborative_filtering/blob/master/NGCF/NGCF.py):
1. **Xavier Weight Initialization (`_init_weights`)**:
   - Initializes 0-th layer embeddings `user_embedding` ($|\mathcal{U}| \times d_0$) and `item_embedding` ($|\mathcal{I}| \times d_0$).
   - For each layer $k \in \{0, \dots, L-1\}$ with dimensions `weight_size_list = [emb_dim] + weight_size`, initializes:
     - `W_gc_k`, `b_gc_k` for first-order neighbor transformation.
     - `W_bi_k`, `b_bi_k` for second-order element-wise product (`ego_embeddings * side_embeddings`) transformation.
2. **Sparse Node/Laplacian Dropout (`_dropout_sparse`)**:
   - When `node_dropout_flag = 1` during training, drops non-zero entries in `norm_adj` with rate `node_dropout` and rescales surviving entries by `1 / (1 - node_dropout)`.
3. **NGCF Embedding Propagation (`_create_ngcf_embed`)**:
   - Computes first-order (`sum_embeddings`) and second-order (`bi_embeddings`) messages with `LeakyReLU(negative_slope=0.2)`:
     ```python
     side_embeddings = torch.sparse.mm(A_hat, ego_embeddings)
     sum_embeddings = F.leaky_relu(torch.matmul(side_embeddings, W_gc_k) + b_gc_k, negative_slope=0.2)
     bi_embeddings = torch.mul(ego_embeddings, side_embeddings)
     bi_embeddings = F.leaky_relu(torch.matmul(bi_embeddings, W_bi_k) + b_bi_k, negative_slope=0.2)
     ego_embeddings = sum_embeddings + bi_embeddings
     ```
   - Applies message dropout (`F.dropout(ego_embeddings, p=mess_dropout[k], training=self.training)`).
   - Applies $\ell_2$-normalization (`norm_embeddings = F.normalize(ego_embeddings, p=2, dim=1)`) before appending to `all_embeddings`.
   - **Concatenates** all layer representations along `dim=1`: $e_u^* = [e_u^{(0)} \,\|\, \hat{e}_u^{(1)} \,\|\, \dots \,\|\, \hat{e}_u^{(L)}]$.
4. **BPR Loss & Regularization (`create_bpr_loss`)**:
   - Computes pairwise BPR loss `-torch.mean(F.logsigmoid(pos_scores - neg_scores))` and $\ell_2$ regularization on the batch embeddings.

In [ ]:
# ==============================================================================
# 3. NGCF Model Implementation (from xiangwang1223/neural_graph_collaborative_filtering)
# ==============================================================================

class NGCF(nn.Module):
    """
    PyTorch implementation of Neural Graph Collaborative Filtering (Wang et al., SIGIR 2019)
    following the official codebase https://github.com/xiangwang1223/neural_graph_collaborative_filtering
    and the experimental settings in the SGL paper (Section 4.1.1 & Section 4.1.2).
    """

    def __init__(
        self,
        n_users: int,
        n_items: int,
        norm_adj: torch.Tensor,
        emb_dim: int = 64,
        weight_size: list[int] | None = None,
        node_dropout_flag: int = 1,
        node_dropout: float = 0.1,
        mess_dropout: list[float] | None = None,
        decay: float = 1e-4,
    ):
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.norm_adj = norm_adj
        self.emb_dim = emb_dim
        self.weight_size = weight_size if weight_size is not None else [64, 64, 64]
        self.n_layers = len(self.weight_size)
        self.node_dropout_flag = node_dropout_flag
        self.node_dropout = node_dropout
        self.mess_dropout = (
            mess_dropout if mess_dropout is not None else [0.1] * self.n_layers
        )
        self.decay = decay

        # Cached final concatenated representations for fast evaluation
        self._user_embeddings_final = None
        self._item_embeddings_final = None

        # Initialize ego embeddings and per-layer transformation matrices with Xavier uniform
        self.embedding_dict, self.weight_dict = self._init_weights()

    def _init_weights(self) -> tuple[nn.ParameterDict, nn.ParameterDict]:
        """
        Initializes all trainable parameters with Xavier uniform initialization
        (matching `tf.contrib.layers.xavier_initializer()` in NGCF.py and Section 4.1.2 of SGL).
        """
        initializer = nn.init.xavier_uniform_

        embedding_dict = nn.ParameterDict(
            {
                "user_embedding": nn.Parameter(
                    initializer(torch.empty(self.n_users, self.emb_dim))
                ),
                "item_embedding": nn.Parameter(
                    initializer(torch.empty(self.n_items, self.emb_dim))
                ),
            }
        )

        weight_dict = nn.ParameterDict()
        self.weight_size_list = [self.emb_dim] + self.weight_size

        for k in range(self.n_layers):
            d_in, d_out = self.weight_size_list[k], self.weight_size_list[k + 1]
            # First-order (GCN sum) weights and biases
            weight_dict[f"W_gc_{k}"] = nn.Parameter(initializer(torch.empty(d_in, d_out)))
            weight_dict[f"b_gc_{k}"] = nn.Parameter(initializer(torch.empty(1, d_out)))
            # Second-order (Bi-interaction element-wise product) weights and biases
            weight_dict[f"W_bi_{k}"] = nn.Parameter(initializer(torch.empty(d_in, d_out)))
            weight_dict[f"b_bi_{k}"] = nn.Parameter(initializer(torch.empty(1, d_out)))

        return embedding_dict, weight_dict

    def _dropout_sparse(self, x: torch.Tensor, drop_rate: float) -> torch.Tensor:
        """
        Applies sparse node/edge dropout to the normalized Laplacian tensor `x`
        with keep probability `1.0 - drop_rate` and inverted scaling `1.0 / (1.0 - drop_rate)`.
        """
        if drop_rate <= 0.0:
            return x
        x = x.coalesce()
        keep_prob = 1.0 - drop_rate
        indices = x.indices()
        values = x.values()

        random_tensor = keep_prob + torch.rand(values.size(0), device=values.device)
        dropout_mask = torch.floor(random_tensor).bool()

        kept_indices = indices[:, dropout_mask]
        kept_values = values[dropout_mask] / keep_prob
        return torch.sparse_coo_tensor(
            kept_indices, kept_values, x.size(), device=values.device
        ).coalesce()

    def _create_ngcf_embed(self, drop_flag: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Executes L layers of NGCF message passing with second-order feature interaction
        and concatenates all layer representations [E^{(0)} || E^{(1)} || ... || E^{(L)}].
        Directly mirrors `_create_ngcf_embed` in `NGCF/NGCF.py`.
        """
        if self.node_dropout_flag and drop_flag and self.training:
            A_hat = self._dropout_sparse(self.norm_adj, self.node_dropout)
        else:
            A_hat = self.norm_adj

        ego_embeddings = torch.cat(
            [self.embedding_dict["user_embedding"], self.embedding_dict["item_embedding"]],
            dim=0,
        )
        all_embeddings = [ego_embeddings]

        for k in range(self.n_layers):
            # 1. Aggregate messages from neighbors: side_embeddings = A_hat * E^{(k)}
            side_embeddings = torch.sparse.mm(A_hat, ego_embeddings)

            # 2. First-order transformed sum messages of neighbors
            sum_embeddings = F.leaky_relu(
                torch.matmul(side_embeddings, self.weight_dict[f"W_gc_{k}"])
                + self.weight_dict[f"b_gc_{k}"],
                negative_slope=0.2,
            )

            # 3. Second-order bi-interaction messages: element-wise product E^{(k)} * (A_hat * E^{(k)})
            bi_embeddings = torch.mul(ego_embeddings, side_embeddings)
            bi_embeddings = F.leaky_relu(
                torch.matmul(bi_embeddings, self.weight_dict[f"W_bi_{k}"])
                + self.weight_dict[f"b_bi_{k}"],
                negative_slope=0.2,
            )

            # 4. Combine first-order and second-order messages
            ego_embeddings = sum_embeddings + bi_embeddings

            # 5. Message dropout
            if self.mess_dropout[k] > 0.0:
                ego_embeddings = F.dropout(
                    ego_embeddings, p=self.mess_dropout[k], training=self.training
                )

            # 6. L2-normalize the distribution of embeddings before appending to readout list
            norm_embeddings = F.normalize(ego_embeddings, p=2, dim=1)
            all_embeddings.append(norm_embeddings)

        # 7. Concatenation readout across layers 0, 1, ..., L (shape: |V| x sum(d_l))
        all_embeddings = torch.cat(all_embeddings, dim=1)
        u_g_embeddings, i_g_embeddings = torch.split(
            all_embeddings, [self.n_users, self.n_items], dim=0
        )
        return u_g_embeddings, i_g_embeddings

    def forward(
        self,
        users: torch.Tensor,
        pos_items: torch.Tensor,
        neg_items: torch.Tensor,
        drop_flag: bool = True,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Computes concatenated NGCF embeddings for a mini-batch of (users, pos_items, neg_items)."""
        ua_embeddings, ia_embeddings = self._create_ngcf_embed(drop_flag=drop_flag)
        u_g_embeddings = ua_embeddings[users]
        pos_i_g_embeddings = ia_embeddings[pos_items]
        neg_i_g_embeddings = ia_embeddings[neg_items]
        return u_g_embeddings, pos_i_g_embeddings, neg_i_g_embeddings

    def create_bpr_loss(
        self,
        users_emb: torch.Tensor,
        pos_items_emb: torch.Tensor,
        neg_items_emb: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Computes the pairwise BPR loss and L2 regularization loss exactly as in `NGCF/NGCF.py`:
            mf_loss  = -mean(log(sigmoid(pos_scores - neg_scores)))
            emb_loss = decay * 0.5 * (||u||_2^2 + ||i||_2^2 + ||j||_2^2) / batch_size
        """
        batch_size = users_emb.size(0)
        pos_scores = torch.sum(torch.mul(users_emb, pos_items_emb), dim=1)
        neg_scores = torch.sum(torch.mul(users_emb, neg_items_emb), dim=1)

        maxi = F.logsigmoid(pos_scores - neg_scores)
        mf_loss = -torch.mean(maxi)

        regularizer = 0.5 * (
            users_emb.norm(2).pow(2)
            + pos_items_emb.norm(2).pow(2)
            + neg_items_emb.norm(2).pow(2)
        )
        emb_loss = self.decay * (regularizer / float(batch_size))

        total_loss = mf_loss + emb_loss
        return total_loss, mf_loss, emb_loss

    def eval(self):
        """Switches model to eval mode and caches deterministic concatenated embeddings."""
        super().eval()
        with torch.no_grad():
            self._user_embeddings_final, self._item_embeddings_final = self._create_ngcf_embed(
                drop_flag=False
            )
        return self

    @torch.no_grad()
    def predict(self, users: np.ndarray | list[int] | torch.Tensor) -> np.ndarray:
        """
        Adapter method for the shared `Evaluator`: computes full-ranking inner-product scores
        y_hat_{ui} = (e_u^*)^T e_i^* across all items and returns a NumPy array.
        """
        if not isinstance(users, torch.Tensor):
            device = self.embedding_dict["user_embedding"].device
            users = torch.from_numpy(np.asarray(users)).long().to(device)
        if self._user_embeddings_final is None or self._item_embeddings_final is None:
            self._user_embeddings_final, self._item_embeddings_final = self._create_ngcf_embed(
                drop_flag=False
            )
        u_g_embeddings = self._user_embeddings_final[users.long()]
        ratings = torch.matmul(u_g_embeddings, self._item_embeddings_final.t())
        return ratings.cpu().detach().numpy()

## 4. All-Ranking Evaluation Protocol & Long-Tail Group Analysis (`Evaluator` — Shared Pipeline)

Uses the exact same `Evaluator` and `compute_user_metrics` implementation as `SGL_Colab.ipynb` and `LightGCN_Colab.ipynb`:
- Masks out training items (`-np.inf`) and evaluates `Precision@20`, `Recall@20`, and `NDCG@20` over all items.
- Computes the 10-group Long-Tail Item Popularity `Recall@20` decomposition.

In [ ]:
# ==============================================================================
# 4. Top-K All-Ranking Evaluator & Long-Tail Group Decomposition (Shared)
# ==============================================================================

def compute_user_metrics(
    ranked_items: np.ndarray,
    ground_truth: set[int],
    top_k_list: list[int],
) -> dict[str, np.ndarray]:
    """
    Computes Precision@K, Recall@K, and NDCG@K for a single user u.
    """
    max_k = len(ranked_items)
    truth_len = len(ground_truth)

    hits = np.array(
        [1.0 if item in ground_truth else 0.0 for item in ranked_items], dtype=np.float64
    )
    cum_hits = np.cumsum(hits)
    ranks = np.arange(1, max_k + 1, dtype=np.float64)

    precisions = cum_hits / ranks
    recalls = cum_hits / truth_len if truth_len > 0 else np.zeros_like(cum_hits)

    discounts = 1.0 / np.log2(np.arange(2, max_k + 2, dtype=np.float64))
    dcg = np.cumsum(hits * discounts)

    ideal_hits = np.zeros(max_k, dtype=np.float64)
    ideal_hits[: min(max_k, truth_len)] = 1.0
    idcg = np.cumsum(ideal_hits * discounts)
    ndcgs = np.where(idcg > 0, dcg / idcg, 0.0)

    k_indices = [k - 1 for k in top_k_list]
    return {
        "Precision": precisions[k_indices],
        "Recall": recalls[k_indices],
        "NDCG": ndcgs[k_indices],
    }


class Evaluator:
    """All-ranking Top-K Evaluator (shared with SGL_Colab.ipynb & LightGCN_Colab.ipynb)."""

    def __init__(
        self,
        user_train_dict: dict[int, np.ndarray],
        user_test_dict: dict[int, np.ndarray],
        metrics: list[str] | None = None,
        top_k: int | list[int] = 20,
        batch_size: int = 128,
    ):
        if metrics is None:
            metrics = ["Precision", "Recall", "NDCG"]
        elif isinstance(metrics, str):
            metrics = [metrics]

        self.user_pos_train = user_train_dict if user_train_dict is not None else {}
        self.user_pos_test = user_test_dict
        self.metrics = metrics
        self.batch_size = batch_size
        self.top_show = [top_k] if isinstance(top_k, int) else sorted(list(top_k))
        self.max_top = max(self.top_show)

    def metrics_info(self) -> str:
        headers = []
        for k in self.top_show:
            for m in self.metrics:
                headers.append(f"{m}@{k}".ljust(12))
        return "\t".join(headers)

    def evaluate(self, model, test_users: list[int] | None = None) -> tuple[np.ndarray, str]:
        model.eval()
        if test_users is None:
            test_users = [u for u, items in self.user_pos_test.items() if len(items) > 0]

        all_user_results = []

        for start in range(0, len(test_users), self.batch_size):
            batch_users = test_users[start : start + self.batch_size]
            ranking_score = model.predict(batch_users)
            ranking_score = np.array(ranking_score, dtype=np.float32, copy=True)

            # Mask observed training items with -inf so they are never recommended
            for idx, u in enumerate(batch_users):
                if u in self.user_pos_train and len(self.user_pos_train[u]) > 0:
                    ranking_score[idx, self.user_pos_train[u]] = -np.inf

            top_k_part = np.argpartition(-ranking_score, self.max_top - 1, axis=1)[:, : self.max_top]
            row_indices = np.arange(len(batch_users))[:, None]
            top_k_scores = ranking_score[row_indices, top_k_part]
            top_k_order = np.argsort(-top_k_scores, axis=1)
            top_k_items = top_k_part[row_indices, top_k_order]

            for idx, u in enumerate(batch_users):
                truth_set = set(self.user_pos_test[u].tolist())
                user_metric_dict = compute_user_metrics(
                    ranked_items=top_k_items[idx],
                    ground_truth=truth_set,
                    top_k_list=self.top_show,
                )
                user_vec = []
                for k_idx in range(len(self.top_show)):
                    for m in self.metrics:
                        user_vec.append(user_metric_dict[m][k_idx])
                all_user_results.append(user_vec)

        all_user_results = np.asarray(all_user_results, dtype=np.float64)
        final_result = np.mean(all_user_results, axis=0)
        buf = "\t".join([("%.8f" % x).ljust(12) for x in final_result])
        return final_result, buf

    def evaluate_long_tail_groups(
        self,
        model,
        train_items: np.ndarray,
        num_items: int,
        num_groups: int = 10,
        k: int = 20,
    ) -> np.ndarray:
        """
        Decomposes Recall@K across 10 item popularity groups (Section 4.3.1, Figure 4).
        Group 1 = lowest-degree (long-tail) items; Group 10 = highest-degree (head) items.
        """
        model.eval()
        item_degrees = np.bincount(train_items, minlength=num_items)
        sorted_items = np.argsort(item_degrees)
        sorted_degrees = item_degrees[sorted_items]
        cum_degrees = np.cumsum(sorted_degrees)
        total_interactions = cum_degrees[-1]

        item_to_group = np.zeros(num_items, dtype=np.int32)
        thresholds = np.linspace(0, total_interactions, num_groups + 1)[1:]
        group_ids = np.clip(np.searchsorted(thresholds, cum_degrees, side="left"), 0, num_groups - 1)
        item_to_group[sorted_items] = group_ids

        test_users = [u for u, items in self.user_pos_test.items() if len(items) > 0]
        group_recalls = np.zeros(num_groups, dtype=np.float64)

        for start in range(0, len(test_users), self.batch_size):
            batch_users = test_users[start : start + self.batch_size]
            ranking_score = np.array(model.predict(batch_users), dtype=np.float32, copy=True)

            for idx, u in enumerate(batch_users):
                if u in self.user_pos_train and len(self.user_pos_train[u]) > 0:
                    ranking_score[idx, self.user_pos_train[u]] = -np.inf

            top_k_part = np.argpartition(-ranking_score, k - 1, axis=1)[:, :k]

            for idx, u in enumerate(batch_users):
                truth_set = set(self.user_pos_test[u].tolist())
                truth_len = len(truth_set)
                for rec_item in top_k_part[idx]:
                    if rec_item in truth_set:
                        g = item_to_group[int(rec_item)]
                        group_recalls[g] += 1.0 / truth_len

        group_recalls /= max(1, len(test_users))
        return group_recalls

## 5. Configuration, Training Pipeline & Visualization (`NGCFConfig`)

`NGCFConfig` below is configured with the exact experimental settings described in **Section 4.1.1 & Section 4.1.2 of the SGL paper**:
- **Initializer**: Xavier Uniform (`xavier_uniform_`).
- **Optimizer**: Adam with `lr = 0.001` and `batch_size = 2048`.
- **Architecture**: `embed_size = 64`, 3 GCN layers `layer_size = [64, 64, 64]`, `reg = 1e-4` ($\lambda_2$), `node_dropout = 0.1`, and `mess_dropout = [0.1, 0.1, 0.1]`.

In [ ]:
# ==============================================================================
# 5. Hyperparameter Configuration & End-to-End NGCF Training / Evaluation
# ==============================================================================

@dataclass
class NGCFConfig:
    # Dataset settings (loads ONLY `<dataset>.train` and splits 7:1:2 into train:val:test)
    data_dir: str = "./dataset/yelp2018"
    dataset: str = "yelp2018"
    split_ratios: tuple[float, float, float] = (0.7, 0.1, 0.2)
    seed: int = 2021

    # NGCF Model Hyperparameters (matching SGL paper Section 4.1.2 & NGCF parser.py)
    embed_size: int = 64                                            # d = 64 in SGL paper
    layer_size: list[int] = field(default_factory=lambda: [64, 64, 64])  # 3 propagation layers
    reg: float = 1e-4                                               # L2 regularization weight (regs=[1e-4])
    adj_type: str = "norm_adj"                                      # 'norm_adj' (D^-1(A+I)) or 'sym_adj' (D^-0.5 A D^-0.5)
    node_dropout_flag: int = 1                                      # 1: activate node dropout, 0: disable
    node_dropout: float = 0.1     # Keep prob = 1 - 0.1 = 0.9
    mess_dropout: list[float] = field(default_factory=lambda: [0.1, 0.1, 0.1])  # Message dropout per layer

    # Optimization & Evaluation settings (SGL paper Section 4.1.2: Adam lr=0.001, batch_size=2048)
    lr: float = 0.001
    batch_size: int = 2048
    epochs: int = 30
    verbose: int = 1                # Evaluate on Val split every `verbose` epochs
    stop_cnt: int = 10              # Early stopping patience on Val Recall@20
    top_k: list[int] = field(default_factory=lambda: [20])
    test_batch_size: int = 128
    start_testing_epoch: int = 0

    def to_ngcf_args(self) -> dict:
        """Converts dataclass to the `args` dictionary expected by `NGCF.__init__`."""
        return {
            "embed_size": self.embed_size,
            "layer_size": self.layer_size,
            "regs": [self.reg],
            "node_dropout_flag": self.node_dropout_flag,
            "node_dropout": self.node_dropout,
            "mess_dropout": self.mess_dropout,
            "batch_size": self.batch_size,
            "Ks": self.top_k,
        }


def run_ngcf_pipeline(cfg: NGCFConfig):
    """Runs the complete NGCF preprocessing (7:1:2 split), training, val/test evaluation, and plotting pipeline."""
    set_seed(cfg.seed)

    print("=" * 80)
    print("Neural Graph Collaborative Filtering (NGCF) for Recommendation (SIGIR '19)")
    print("=" * 80)
    print(
        f"[Config] dataset={cfg.dataset} | embed_size={cfg.embed_size} | "
        f"layer_size={cfg.layer_size} | adj_type={cfg.adj_type} | lr={cfg.lr} | "
        f"reg={cfg.reg} | batch_size={cfg.batch_size}"
    )
    print(
        f"[Drop]   node_dropout_flag={cfg.node_dropout_flag} (ratio={cfg.node_dropout}) | "
        f"mess_dropout={cfg.mess_dropout}"
    )
    print(f"[Device] Using {DEVICE}")

    # 1. Load `<dataset>.train` ONLY, split 7:1:2 into train/val/test, and build graph G on 70% train
    dataset = SGLDataset(
        data_dir=cfg.data_dir,
        dataset_name=cfg.dataset,
        adj_type=cfg.adj_type,
        split_ratios=cfg.split_ratios,
        seed=cfg.seed,
        device=DEVICE,
    )
    print(
        f"[Data]   #Users: {dataset.num_users} | #Items: {dataset.num_items} | "
        f"#Total Interactions (from {cfg.dataset}.train): {dataset.total_interactions}"
    )
    print(
        f"[Split]  train:val:test = {cfg.split_ratios[0]:.0%}:{cfg.split_ratios[1]:.0%}:{cfg.split_ratios[2]:.0%} "
        f"(7:1:2) -> #Train: {dataset.num_ratings} | #Val: {dataset.num_val_ratings} | #Test: {dataset.num_test_ratings}"
    )

    data_iter = PairwiseSampler(dataset, batch_size=cfg.batch_size, shuffle=True)

    # Validation evaluator: masks 70% train items, evaluates on 10% validation items
    val_evaluator = Evaluator(
        user_train_dict=dataset.train_user_dict,
        user_test_dict=dataset.val_user_dict,
        metrics=["Precision", "Recall", "NDCG"],
        top_k=cfg.top_k,
        batch_size=cfg.test_batch_size,
    )
    # Test evaluator: masks 80% (train + val) items, evaluates on 20% held-out test items
    test_evaluator = Evaluator(
        user_train_dict=dataset.train_val_user_dict,
        user_test_dict=dataset.test_user_dict,
        metrics=["Precision", "Recall", "NDCG"],
        top_k=cfg.top_k,
        batch_size=cfg.test_batch_size,
    )

    # 2. Initialize NGCF model & Adam optimizer
    model = NGCF(
        n_users=dataset.num_users,
        n_items=dataset.num_items,
        norm_adj=dataset.norm_adj,
        emb_dim=cfg.embed_size,
        weight_size=cfg.layer_size,
        node_dropout_flag=cfg.node_dropout_flag,
        node_dropout=cfg.node_dropout,
        mess_dropout=cfg.mess_dropout,
        decay=cfg.reg,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    print("-" * 80)
    print("Val Metrics:	" + val_evaluator.metrics_info())
    print("-" * 80)

    best_epoch = 0
    best_val_result = np.zeros(3 * len(cfg.top_k), dtype=np.float64)
    best_state_dict = None
    stopping_step = 0

    history = {"epoch": [], "bpr_loss": [], "reg_loss": [], "recall": [], "ndcg": []}

    # 3. Training & Validation Loop
    for epoch in range(1, cfg.epochs + 1):
        total_loss, total_bpr_loss, total_reg_loss = 0.0, 0.0, 0.0
        num_batches = 0
        t0 = time()

        model.train()
        for bat_users, bat_pos_items, bat_neg_items in data_iter:
            bat_users = torch.from_numpy(bat_users).long().to(DEVICE)
            bat_pos_items = torch.from_numpy(bat_pos_items).long().to(DEVICE)
            bat_neg_items = torch.from_numpy(bat_neg_items).long().to(DEVICE)

            u_g_emb, pos_i_g_emb, neg_i_g_emb = model(
                bat_users, bat_pos_items, bat_neg_items, drop_flag=bool(cfg.node_dropout_flag)
            )
            loss, mf_loss, emb_loss = model.create_bpr_loss(
                u_g_emb, pos_i_g_emb, neg_i_g_emb
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_bpr_loss += mf_loss.item()
            total_reg_loss += emb_loss.item()
            num_batches += 1

        avg_total = total_loss / max(1, num_batches)
        avg_bpr = total_bpr_loss / max(1, num_batches)
        avg_reg = total_reg_loss / max(1, num_batches)

        print(
            f"[iter {epoch:2d} : loss : {avg_total:.4f} = {avg_bpr:.4f} (BPR) + "
            f"{avg_reg:.4f} (Reg), time: {time() - t0:.2f}s]"
        )

        if epoch % cfg.verbose == 0 and epoch > cfg.start_testing_epoch:
            val_result, buf = val_evaluator.evaluate(model)
            print(f"epoch {epoch:2d} [Val]:	{buf}")

            history["epoch"].append(epoch)
            history["bpr_loss"].append(avg_bpr)
            history["reg_loss"].append(avg_reg)
            history["recall"].append(val_result[1])
            history["ndcg"].append(val_result[2])

            if best_val_result[1] < val_result[1]:
                best_val_result = val_result
                best_epoch = epoch
                stopping_step = 0
                best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                stopping_step += 1
                if stopping_step >= cfg.stop_cnt:
                    print(f"Early stopping triggered at epoch: {epoch}")
                    break

    # 4. Restore best validation model & evaluate on the 20% Held-Out Test Split
    if best_state_dict is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state_dict.items()})

    test_result, _ = test_evaluator.evaluate(model)
    group_recalls = test_evaluator.evaluate_long_tail_groups(
        model=model,
        train_items=dataset.train_items,
        num_items=dataset.num_items,
        num_groups=10,
        k=cfg.top_k[0],
    )

    print("=" * 80)
    print(f"Best Validation Result @ Epoch {best_epoch} (10% Val Split):")
    print("Metrics:	" + val_evaluator.metrics_info())
    print("Values: 	" + "\t".join([("%.8f" % x).ljust(12) for x in best_val_result]))
    print("-" * 80)
    print(f"Final Test Result @ Best Epoch {best_epoch} (20% Test Split):")
    print("Metrics:	" + test_evaluator.metrics_info())
    print("Values: 	" + "\t".join([("%.8f" % x).ljust(12) for x in test_result]))
    print("-" * 80)
    group_str = " | ".join([f"G{g+1}: {val:.4f}" for g, val in enumerate(group_recalls)])
    print(f"Test Long-Tail Group Recall@{cfg.top_k[0]}: {group_str}")
    print("=" * 80)

    # 5. Plot Training Curves & Test Long-Tail Group Recall
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

    axes[0].plot(history["epoch"], history["bpr_loss"], marker="o", label="BPR Loss")
    axes[0].plot(history["epoch"], history["reg_loss"], marker="s", label="L2 Reg Loss")
    axes[0].set_title("NGCF Training Losses per Epoch")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Batch-Averaged Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["epoch"], history["recall"], marker="o", color="tab:green", label=f"Val Recall@{cfg.top_k[0]}")
    axes[1].plot(history["epoch"], history["ndcg"], marker="^", color="tab:purple", label=f"Val NDCG@{cfg.top_k[0]}")
    axes[1].set_title(f"NGCF Top-{cfg.top_k[0]} Validation Performance")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Metric Value")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    axes[2].bar(np.arange(1, 11), group_recalls, color="tab:red", alpha=0.85)
    axes[2].set_title(f"NGCF Test Long-Tail Group Recall@{cfg.top_k[0]} (Fig. 4)")
    axes[2].set_xlabel("GroupID (1 = Long-Tail, 10 = Popular Head)")
    axes[2].set_ylabel("Group Recall Contribution")
    axes[2].set_xticks(np.arange(1, 11))
    axes[2].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    return model, history, group_recalls


# Execute the NGCF pipeline with the SGL paper's experimental setup
config = NGCFConfig(
    embed_size=64,
    layer_size=[64, 64, 64],   # 3 GCN layers
    reg=1e-4,                  # Regularization coefficient lambda_2
    lr=0.001,                  # Adam learning rate in SGL paper (Section 4.1.2)
    batch_size=2048,           # Mini-batch size in SGL paper (Section 4.1.2)
    epochs=100,
)
trained_ngcf_model, ngcf_history, ngcf_long_tail_recalls = run_ngcf_pipeline(config)
